<a href="https://colab.research.google.com/github/Shineii86/LeechBot/blob/main/notebooks/LeechBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
<img src="https://capsule-render.vercel.app/api?type=waving&height=300&color=gradient&text=𝗟𝗲𝗲𝗰𝗵%20𝗕𝗼𝘁&fontAlignY=30&fontSize=100&desc=𝖠𝖽𝗏𝖺𝗇𝖼𝖾𝖽%20𝖳𝖾𝗅𝖾𝗀𝗋𝖺𝗆%20𝖥𝗂𝗅𝖾%20𝖳𝗋𝖺𝗇𝗌𝗅𝗈𝖺𝖽𝖾𝗋&descSize=30" />

**A powerful Pyrogram-based bot to transfer files to Telegram & Google Drive**

![Version](https://img.shields.io/badge/Version-3.1.15-8B5CF6?style=for-the-badge)
![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=for-the-badge&logo=python&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-06B6D4?style=for-the-badge)

---

### ✨ Features

| 📥 Download From | 📤 Upload To | 🛠️ Tools |
|:---:|:---:|:---:|
| YouTube, Facebook, Instagram | Telegram | Video Converter (GPU) |
| Google Drive, Mega, Terabox | Google Drive | Archive Handler |
| Pixeldrain, Mediafire, Direct | Directory Leech | Smart Splitting |
| 2000+ sites via yt-dlp | Batch Photos | Download Queue |

---

### 🚀 Quick Start

1. **Fill credentials** in Cell 2 (or use Colab Secrets)
2. Click **Runtime → Run all** or press **Ctrl+F9**
3. Bot starts automatically — send `/start` on Telegram

---

### 📋 Cells

| # | Cell | Purpose |
|:--|:-----|:--------|
| 2 | 📦 Setup LeechBot | Clone repo, install deps, configure |
| 3 | 🚀 Deploy LeechBot | Start bot with keep-alive |

</div>

In [ ]:
# @title 📦 Setup LeechBot
#@markdown <div align="center">
#@markdown <img src="https://user-images.githubusercontent.com/125879861/255391401-371f3a64-732d-4954-ac0f-4f093a6605e1.png" width="500">
#@markdown </div>

#@markdown ---
#@markdown ## 🔐 Credentials
#@markdown > Fill manually or set in **🔑 Secrets** (left panel) with these names:
#@markdown >
#@markdown > `LEECHBOT_API_ID` · `LEECHBOT_API_HASH` · `LEECHBOT_BOT_TOKEN` · `LEECHBOT_USER_ID` · `LEECHBOT_DUMP_ID`

API_ID = 0 # @param {type:"integer"}
API_HASH = "" # @param {type:"string"}
BOT_TOKEN = "" # @param {type:"string"}
OWNER_ID = 0 # @param {type:"integer"}
DUMP_ID = 0 # @param {type:"integer"}

#@markdown ---
#@markdown ## ⚙️ Options
MOUNT_DRIVE = False # @param {type:"boolean"}
USE_GPU = True # @param {type:"boolean"}
REPO_BRANCH = "main" # @param ["main"]

# ═══════════════════════════════════════════════════════════
# 📦 Setup Engine
# ═══════════════════════════════════════════════════════════

import subprocess, sys, os, json, time, shutil
from pathlib import Path
from IPython.display import clear_output, display, Markdown
import logging

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["ALSA_CONFIG_PATH"] = "/dev/null"

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger("LeechBot")

# ─── UI Helpers ───────────────────────────────────────────
STEP = [0]

def banner():
    return """
    ╔═══════════════════════════════════════════╗
    ║         🚀 L E E C H B O T               ║
    ║    Advanced Telegram File Transloader     ║
    ╠═══════════════════════════════════════════╣
    ║  👤 Shinei Nouzen  ·  📂 Shineii86       ║
    ╚═══════════════════════════════════════════╝
    """

def log(emoji, msg, color="#2196F3"):
    display(Markdown(f'<font color="{color}">**{emoji} {msg}**</font>'))

def step(msg):
    STEP[0] += 1
    display(Markdown(f"\n---\n### Step {STEP[0]}: {msg}"))

def ok(msg):   log("✅", msg, "#4CAF50")
def fail(msg): log("❌", msg, "#F44336")
def warn(msg): log("⚠️", msg, "#FF9800")
def info(msg): log("ℹ️", msg, "#2196F3")
def check(label, ok_flag, detail=""):
    icon = "✅" if ok_flag else "❌"
    suffix = f" — `{detail}`" if detail else ""
    display(Markdown(f"{icon} **{label}**{suffix}"))

def run(cmd, desc, retries=3):
    for i in range(retries):
        try:
            info(f"{desc} (attempt {i+1}/{retries})")
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True, timeout=300)
            ok(f"{desc} — done")
            return True
        except subprocess.CalledProcessError as e:
            if i == retries - 1:
                fail(f"{desc} failed: {e.stderr[:200]}")
                return False
            time.sleep(2 ** i)
        except subprocess.TimeoutExpired:
            if i == retries - 1:
                fail(f"{desc} timed out")
                return False
    return False

# ─── Credentials ──────────────────────────────────────────
def load_credentials():
    creds = {}
    try:
        from google.colab import userdata
        secrets = {
            'API_ID': 'LEECHBOT_API_ID', 'API_HASH': 'LEECHBOT_API_HASH',
            'BOT_TOKEN': 'LEECHBOT_BOT_TOKEN', 'OWNER_ID': 'LEECHBOT_USER_ID',
            'DUMP_ID': 'LEECHBOT_DUMP_ID'
        }
        for key, name in secrets.items():
            try:
                val = userdata.get(name)
                creds[key] = int(val) if key in ['API_ID', 'OWNER_ID', 'DUMP_ID'] else val
                ok(f"{key} loaded from Colab Secrets")
            except:
                creds[key] = None
    except ImportError:
        pass

    fallbacks = {'API_ID': API_ID, 'API_HASH': API_HASH, 'BOT_TOKEN': BOT_TOKEN,
                 'OWNER_ID': OWNER_ID, 'DUMP_ID': DUMP_ID}
    for k in fallbacks:
        if not creds.get(k):
            creds[k] = fallbacks[k]
    return creds

def validate(creds):
    required = ['API_ID', 'API_HASH', 'BOT_TOKEN', 'OWNER_ID', 'DUMP_ID']
    missing = [k for k in required if not creds.get(k)]
    if missing:
        fail(f"Missing: {', '.join(missing)}")
        return False
    d = str(creds['DUMP_ID'])
    if len(d) == 10 and not d.startswith('-100'):
        creds['DUMP_ID'] = int(f"-100{d}")
        info("Auto-formatted DUMP_ID with -100 prefix")
    return True

# ─── Setup ───────────────────────────────────────────────
def setup():
    clear_output(wait=True)
    print(banner())

    # Step 1: Credentials
    step("🔐 Load Credentials")
    creds = load_credentials()
    if not validate(creds):
        fail("Fix credentials and re-run this cell")
        return

    # Step 2: Clone
    step("📦 Clone Repository")
    os.chdir("/content")
    if os.path.exists("/content/leechbot"):
        # Persist credentials across re-clones
        _cred_backup = "/content/.leechbot_creds.json"
        _existing_creds = os.path.join("/content/leechbot", "credentials.json")
        if os.path.exists(_existing_creds):
            import shutil as _shutil
            _shutil.copy2(_existing_creds, _cred_backup)
            info("Backed up existing credentials")

        shutil.rmtree("/content/leechbot")
        info("Cleaned previous install")

    if not run(f"git clone -b {REPO_BRANCH} --depth 1 https://github.com/Shineii86/LeechBot.git /content/leechbot",
               "Cloning LeechBot"):
        return

    # Step 3: Dependencies
    step("📦 Install Dependencies")
    if not run("apt-get update -qq && apt-get install -y -qq ffmpeg aria2 megatools p7zip-full unzip",
               "System packages"):
        return
    if not run("pip3 install -q --no-cache-dir -r /content/leechbot/requirements.txt",
               "Python packages"):
        return

    # Install libtorrent for torrent/magnet support (optional)
    step("🧲 Torrent Support")
    _lt_installed = False
    # Primary: apt (works on Colab and most Linux)
    if run("apt-get install -y -qq python3-libtorrent", "libtorrent (apt)", retries=1):
        _lt_installed = True
    # Fallback: conda (if available)
    elif run("conda install -y -q -c conda-forge libtorrent 2>/dev/null", "libtorrent (conda)", retries=1):
        _lt_installed = True
    if _lt_installed:
        ok("libtorrent installed — magnet/torrent support enabled")
    else:
        warn("libtorrent not available — torrent downloads will use aria2c fallback")

    # Step 4: GPU Check
    step("🎮 Hardware Check")
    if USE_GPU:
        try:
            r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                              shell=True, capture_output=True, text=True, check=True)
            name, mem = r.stdout.strip().split(', ')
            ok(f"GPU: {name} ({mem} VRAM)")
        except:
            info("No GPU — using CPU")
    else:
        info("GPU disabled by user")

    # Step 5: Save config
    step("💾 Save Configuration")
    cfg_path = "/content/leechbot/credentials.json"
    with open(cfg_path, 'w') as f:
        json.dump(creds, f, indent=2)
    os.chmod(cfg_path, 0o600)
    ok("credentials.json saved")

    os.environ["API_ID"] = str(creds["API_ID"])
    os.environ["API_HASH"] = str(creds["API_HASH"])
    os.environ["BOT_TOKEN"] = str(creds["BOT_TOKEN"])
    os.environ["OWNER_ID"] = str(creds["OWNER_ID"])
    os.environ["DUMP_ID"] = str(creds["DUMP_ID"])
    ok("Environment variables set")

    # Step 6: Mount Drive (optional)
    if MOUNT_DRIVE:
        step("☁️ Google Drive")
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            ok("Drive mounted")
            try:
                import pickle
                from google.colab import auth
                import google.auth
                from google.auth.transport.requests import Request
                auth.authenticate_user()
                creds_g, _ = google.auth.default()
                if creds_g.expired and creds_g.refresh_token:
                    creds_g.refresh(Request())
                with open('/content/token.pickle', 'wb') as f:
                    pickle.dump(creds_g, f)
                os.chmod('/content/token.pickle', 0o600)
                ok("GDrive token generated")
            except Exception as e:
                warn(f"Token gen skipped: {e}")
        except Exception as e:
            warn(f"Drive mount failed: {e}")

    # Step 7: Clean sessions
    step("🧹 Clean Sessions")
    for sf in ["/content/leechbot/leechbot_session.session",
               "/content/leechbot/leechbot_session.session-journal"]:
        if os.path.exists(sf):
            os.remove(sf)
            info(f"Removed: {sf}")
    ok("Sessions cleaned")

    # Step 8: Health Check
    step("🔍 Health Check")
    v = sys.version.split()[0]
    check("Python 3.10+", tuple(int(x) for x in v.split('.')) >= (3, 10), v)
    try:
        subprocess.run("ffmpeg -version", shell=True, capture_output=True, check=True)
        check("ffmpeg", True)
    except: check("ffmpeg", False, "apt install ffmpeg")
    try:
        subprocess.run("aria2c --version", shell=True, capture_output=True, check=True)
        check("aria2c", True)
    except: check("aria2c", False, "apt install aria2")
    try:
        subprocess.run("megadl --version", shell=True, capture_output=True, check=True)
        check("megatools", True)
    except: check("megatools", False, "apt install megatools")
    try:
        import libtorrent
        check("libtorrent", True)
    except: check("libtorrent", False, "optional — torrent support")
    try:
        import yt_dlp
        check("yt-dlp", True)
    except: check("yt-dlp", False, "pip install yt-dlp")
    try:
        du = shutil.disk_usage("/content")
        free_gb = du.free / (1024**3)
        check("Disk Space", free_gb > 5, f"{free_gb:.1f} GB free")
    except: pass

    ok("Setup complete! Run the **🚀 Deploy** cell next.")

# ─── Run ──────────────────────────────────────────────────
try:
    setup()
except KeyboardInterrupt:
    warn("Cancelled by user")
except Exception as e:
    fail(f"Unexpected error: {e}")
    logger.exception("Full traceback:")

In [ ]:
# @title 🚀 Deploy LeechBot
#@markdown > Run the **📦 Setup** cell first!

#@markdown ---
#@markdown ## ⚙️ Options
ACTION = "Start Bot" # @param ["Start Bot", "Update & Restart", "Stop Bot"]
AUTO_RESTART = True # @param {type:"boolean"}

# ═══════════════════════════════════════════════════════════
# 🚀 Deploy + Keep-Alive Engine
# ═══════════════════════════════════════════════════════════

import subprocess, sys, os, time, threading, datetime
from IPython.display import clear_output, display, Markdown

BOT_LOG = "/content/leechbot/bot.log"
BOT_DIR = "/content/leechbot"
WEB_PORT = os.environ.get("WEB_PORT", "8080")
bot_proc = None
restart_count = 0
MAX_RESTARTS = 5

# ─── UI Helpers ───────────────────────────────────────────
def banner():
    return """
    ╔═══════════════════════════════════════════╗
    ║         🚀 L E E C H B O T               ║
    ║    Advanced Telegram File Transloader     ║
    ╠═══════════════════════════════════════════╣
    ║  👤 Shinei Nouzen  ·  📂 Shineii86       ║
    ╚═══════════════════════════════════════════╝
    """

def log(emoji, msg, color="#2196F3"):
    display(Markdown(f'<font color="{color}">**{emoji} {msg}**</font>'))

def ok(msg):   log("✅", msg, "#4CAF50")
def fail(msg): log("❌", msg, "#F44336")
def warn(msg): log("⚠️", msg, "#FF9800")
def info(msg): log("ℹ️", msg, "#2196F3")

# ─── Handle ACTION ────────────────────────────────────────
if ACTION == "Update & Restart":
    if not os.path.exists(f"{BOT_DIR}/.git"):
        print("❌ LeechBot not found. Run Setup first.")
    else:
        os.chdir(BOT_DIR)
        print("📥 Pulling latest changes...")
        r = subprocess.run("git pull origin main", shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            output = r.stdout.strip()
            if "Already up to date" in output:
                print("✅ Already up to date!")
            else:
                print(f"✅ Updated:\n```\n{output}\n```")
                subprocess.run("pip3 install -q --no-cache-dir -r requirements.txt",
                              shell=True, capture_output=True)
                print("📦 Dependencies updated")
        else:
            print(f"❌ Git pull failed: {r.stderr[:200]}")

if ACTION == "Stop Bot":
    subprocess.run(["fuser", "-k", f"{WEB_PORT}/tcp"], capture_output=True)
    subprocess.run(["pkill", "-f", "python3 -m leechbot"], capture_output=True)
    print("🛑 Bot stopped")


if ACTION == "Start Bot":
    clear_output(wait=True)
    print(banner())

    if not os.path.exists(f"{BOT_DIR}/leechbot/__init__.py"):
        fail("LeechBot not installed. Run the 📦 Setup cell first.")
    else:
        os.chdir(BOT_DIR)

        # Kill stale processes on port
        subprocess.run(["fuser", "-k", f"{WEB_PORT}/tcp"], capture_output=True)
        time.sleep(1)

        # Start bot
        info("Starting LeechBot...")
        log_fh = open(BOT_LOG, "w")
        bot_proc = subprocess.Popen(
            [sys.executable, "-m", "leechbot"],
            stdout=log_fh,
            stderr=subprocess.STDOUT,
            cwd=BOT_DIR
        )

        # Wait for startup
        startup_ok = False
        for _ in range(60):
            time.sleep(1)
            if bot_proc.poll() is not None:
                fail("Bot crashed on startup!")
                with open(BOT_LOG) as f:
                    print(f.read()[-3000:])
                bot_proc = None
                break
            try:
                with open(BOT_LOG) as f:
                    if "LeechBot started successfully" in f.read():
                        startup_ok = True
                        break
            except:
                pass

        if not startup_ok and bot_proc is not None:
            fail("Bot did not start within 60 seconds")
            with open(BOT_LOG) as f:
                print(f.read()[-3000:])
            bot_proc = None

        if bot_proc is not None:
            clear_output(wait=True)
            print(banner())
            ok("LeechBot is running! Send /start on Telegram.")
            display(Markdown("""
| Command | Action |
|:--------|:-------|
| `/start` | Initialize bot |
| `/tupload` | Leech to Telegram |
| `/gdupload` | Mirror to Google Drive |
| `/ytupload` | YouTube / yt-dlp download |
| `/settings` | Bot preferences |
| `/help` | All commands |
"""))

            # ─── Keep-Alive (JS daemon thread + clear_output) ───
            info("Keep-alive active — this cell stays alive to prevent disconnect")

            def _js_keepalive():
                """JS keep-alive in daemon thread — bypasses blocked event loop."""
                js_code = """
                (function() {
                    // Click runtime connect indicators
                    var btn = document.querySelector('#connect');
                    if (btn) btn.click();
                    var nb = document.querySelector('colab-notebook');
                    if (nb && nb.shadowRoot) {
                        var ind = nb.shadowRoot.querySelector('#runtime-indicator');
                        if (ind) ind.click();
                        var btn2 = nb.shadowRoot.querySelector('#connect');
                        if (btn2) btn2.click();
                    }
                    // Simulate DOM activity
                    window.scrollTo(0, 1); window.scrollTo(0, -1);
                    document.dispatchEvent(new MouseEvent('mousemove', {
                        clientX: Math.random() * window.innerWidth,
                        clientY: Math.random() * window.innerHeight
                    }));
                    document.dispatchEvent(new KeyboardEvent('keydown', {key: 'Shift'}));
                    window.dispatchEvent(new Event('focus'));
                    setTimeout(function(){ window.dispatchEvent(new Event('blur')); }, 100);
                    setTimeout(function(){ window.dispatchEvent(new Event('focus')); }, 200);
                })();
                """
                try:
                    from google.colab import output
                    while True:
                        try: output.eval_js(js_code, timeout_sec=5)
                        except: pass
                        time.sleep(25)
                except ImportError:
                    pass  # Not on Colab

            threading.Thread(target=_js_keepalive, daemon=True).start()

            # ─── Monitor loop (clear_output forces Colab to see activity) ───
            start_time = time.time()

            def get_bot_pid():
                try:
                    r = subprocess.run(["pgrep", "-f", "python3 -m leechbot"],
                                      capture_output=True, text=True)
                    if r.returncode == 0:
                        return int(r.stdout.strip().split()[0])
                except: pass
                return None

            def restart_bot():
                global restart_count, bot_proc
                restart_count += 1
                subprocess.run(["fuser", "-k", f"{WEB_PORT}/tcp"], capture_output=True)
                time.sleep(2)
                fh = open(BOT_LOG, "a")
                fh.write(f"\n\n{'='*40}\nAuto-restart #{restart_count} at {datetime.datetime.now()}\n{'='*40}\n\n")
                fh.close()
                fh = open(BOT_LOG, "a")
                bot_proc = subprocess.Popen(
                    [sys.executable, "-m", "leechbot"],
                    stdout=fh, stderr=subprocess.STDOUT, cwd=BOT_DIR
                )
                time.sleep(10)
                return bot_proc.poll() is None

            try:
                while True:
                    time.sleep(20)
                    elapsed = int(time.time() - start_time)
                    h, m, s = elapsed // 3600, (elapsed % 3600) // 60, elapsed % 60
                    pid = get_bot_pid()
                    ts = datetime.datetime.now().strftime("%H:%M:%S")

                    if pid:
                        last_log = ""
                        try:
                            with open(BOT_LOG) as f:
                                lines = [l.strip() for l in f.readlines() if l.strip()]
                                last_log = lines[-1][:50] if lines else ""
                        except: pass
                        clear_output(wait=True)
                        print(f"{banner()}")
                        print(f"💓 [{ts}] Uptime: {h:02d}:{m:02d}:{s:02d} | PID: {pid}")
                        print(f"📋 Last: {last_log}")
                        print(f"🛡️ Keep-alive: JS thread active | Auto-restart: {AUTO_RESTART}")
                    else:
                        clear_output(wait=True)
                        print(f"{banner()}")
                        print(f"⚠️ [{ts}] Bot process dead! Uptime: {h:02d}:{m:02d}:{s:02d}")
                        if AUTO_RESTART and restart_count < MAX_RESTARTS:
                            if restart_bot():
                                ok("Bot restarted")
                            else:
                                fail("Restart failed")
                        elif restart_count >= MAX_RESTARTS:
                            fail(f"Max restarts ({MAX_RESTARTS}) reached.")
                            break
                        else:
                            fail("Auto-restart disabled.")
                            break
            except KeyboardInterrupt:
                print("\n")
                info("Stopping bot...")
                if bot_proc:
                    bot_proc.terminate()
                    try: bot_proc.wait(timeout=10)
                    except: bot_proc.kill()
                ok("Bot stopped")
